# Module 1: Introduction to ACL2 and Formal Verification

## CS 498: Formal Methods in Computer Science with ACL2

**Course Overview:** This is a university-level course in formal methods using ACL2
(A Computational Logic for Applicative Common Lisp), an industrial-strength theorem
prover that has been used to verify real-world systems including AMD processors,
the Java Virtual Machine, cryptographic algorithms, and operating system kernels.

ACL2 won the **2005 ACM Software System Award** — the same award given to Unix, TeX,
the World Wide Web, and Java.

**What you will learn in this module:**
1. ACL2 as a programming language (based on Lisp)
2. How to define functions with guaranteed termination
3. How to state and prove theorems about programs
4. The verification workflow: model → specify → prove → execute

**Prerequisites:** Basic programming experience. No prior Lisp or formal methods knowledge required.

**Corresponding CSLib/Lean4 topics:** This module covers the foundations that CSLib's
"Pillar 1" aims to formalize — but ACL2 has had these capabilities since the 1990s,
with 400,000+ verified theorems in its community books library.

## 1. ACL2 as a Calculator

ACL2 is based on Common Lisp, so expressions use **prefix notation**: `(+ 2 3)` instead of `2 + 3`.
Every expression is surrounded by parentheses, with the function/operator first.

This is the same notation used in formal logic — it eliminates ambiguity about
operator precedence and associativity.

In [ ]:
; Basic arithmetic — prefix notation
(+ 2 3)

In [ ]:
; Nesting works naturally: compute (3 × 4) + (10 ÷ 2)
(+ (* 3 4) (/ 10 2))

In [ ]:
; ACL2 supports exact rational arithmetic — no floating-point errors
(/ 1 3)

In [ ]:
; Boolean values: t = true, nil = false
(< 3 5)

In [ ]:
; Equality testing
(equal 'hello 'hello)

## 2. Data: Atoms and Lists

ACL2 has several kinds of **atoms** (indivisible values):
- **Numbers:** `42`, `3/7`, `-5`
- **Symbols:** `'hello`, `'x`, `t`, `nil`
- **Characters:** `#\A`, `#\Space`
- **Strings:** `"hello world"`

The fundamental compound data structure is the **list**, built from pairs using `cons`.
`'(1 2 3)` is shorthand for `(cons 1 (cons 2 (cons 3 nil)))`.

In [ ]:
; Build a list
(list 1 2 3)

In [ ]:
; car = first element, cdr = rest of the list
(car '(a b c))

In [ ]:
(cdr '(a b c))

In [ ]:
; cons builds a pair — the fundamental building block
(cons 'x '(y z))

In [ ]:
; consp tests if something is a pair, endp tests for end of list
(consp '(1 2 3))

In [ ]:
; The empty list nil is NOT a cons pair
(consp nil)

## 3. Defining Functions

In ACL2, you define functions with `defun`. Every function must be:

1. **Total** — defined on all possible inputs (no "undefined behavior")
2. **Terminating** — guaranteed to halt on every input (no infinite loops)

ACL2 automatically checks both conditions. If it can't prove termination, it **rejects
the definition**. This is the first layer of trustworthiness: you cannot define a
function that might loop forever.

### How termination works

ACL2 looks for a **measure** — a value that strictly decreases on every recursive call
and is bounded below (like a natural number). For most functions, ACL2 finds this
automatically using `(acl2-count x)` which counts the total cons structure.

In [ ]:
; Length of a list — ACL2 proves termination because (cdr x) is smaller than x
(defun my-len (x)
  (if (endp x)
      0
      (+ 1 (my-len (cdr x)))))

In [ ]:
; Test it
(my-len '(a b c d e))

In [ ]:
; Append two lists
(defun my-app (x y)
  (if (endp x)
      y
      (cons (car x)
            (my-app (cdr x) y))))

In [ ]:
(my-app '(1 2 3) '(4 5 6))

In [ ]:
; Member test — is element e in list x?
(defun my-mem (e x)
  (if (endp x)
      nil
      (or (equal e (car x))
          (my-mem e (cdr x)))))

In [ ]:
(my-mem 'b '(a b c))

In [ ]:
(my-mem 'd '(a b c))

## 4. Proving Theorems — The Heart of Formal Verification

In testing, you check a few inputs:
- `(my-app '(1 2) '(3))` → `(1 2 3)` ✓
- `(my-app nil '(a))` → `(a)` ✓

But there are **infinitely many** possible inputs. How do you know it works on ALL of them?

**`defthm`** lets you state a **universal property** and ACL2 will try to **prove** it
for all possible inputs using:
- **Induction** (structural recursion over data)
- **Rewriting** (applying previously proved theorems)
- **Decision procedures** (for arithmetic, boolean logic, etc.)

### Theorem 1: Appending nil does nothing

In [ ]:
; Prove: for ALL lists x, appending nil returns x (modulo true-list-fix)
; ACL2 discovers this needs induction on x and proves it automatically
(defthm my-app-nil
  (equal (my-app x nil)
         (true-list-fix x)))

### Theorem 2: Append is associative

This is a fundamental algebraic property: `(a ++ b) ++ c = a ++ (b ++ c)`.

In conventional programming, you would *hope* this is true based on testing.
In ACL2, we *prove* it holds for **every** possible combination of lists.

In [ ]:
; Prove: append is associative — for ALL lists x, y, z
(defthm my-app-assoc
  (equal (my-app (my-app x y) z)
         (my-app x (my-app y z))))

### Theorem 3: Length distributes over append

`|x ++ y| = |x| + |y|`

This holds for every pair of lists — not just the ones we tested.

In [ ]:
; Prove: length of (append x y) = length(x) + length(y)
(defthm my-len-of-my-app
  (equal (my-len (my-app x y))
         (+ (my-len x) (my-len y))))

## 5. List Reversal — A Complete Verification Example

Reversing a list seems simple, but bugs in list manipulation have caused real
security vulnerabilities. Let's define reverse, then prove it correct.

In [ ]:
; Reverse using a tail-recursive accumulator
(defun my-rev-aux (x acc)
  (if (endp x)
      acc
      (my-rev-aux (cdr x) (cons (car x) acc))))

(defun my-rev (x)
  (my-rev-aux x nil))

In [ ]:
(my-rev '(1 2 3 4 5))

In [ ]:
; Key lemma: understand the accumulator
(defthm my-rev-aux-is-append
  (equal (my-rev-aux x acc)
         (my-app (my-rev-aux x nil) acc)))

In [ ]:
; THE BIG THEOREM: reversing preserves length — for ALL lists
(defthm my-len-of-my-rev
  (equal (my-len (my-rev x))
         (my-len x)))

## 6. When Proofs Fail — ACL2 as a Bug Finder

If you state something **false**, ACL2 won't just say "I can't prove it."
It will actively search for a **counterexample** to show you why.

In [ ]:
; This is FALSE — reverse doesn't return the same list!
; ACL2 will find a counterexample
(thm (equal (my-rev x) x))

## 7. Guards — Bridging Logic and Execution

ACL2 functions are defined in *logic mode* where they accept all inputs.
**Guards** specify preconditions for efficient execution — once verified,
ACL2 can compile functions to fast native code.

This is how formal verification connects to real systems: the same function
that has mathematical proofs also runs as executable code.

In [ ]:
; Factorial with a guard: only accepts natural numbers
(defun my-fact (n)
  (declare (xargs :guard (natp n)))
  (if (zp n)
      1
      (* n (my-fact (- n 1)))))

In [ ]:
(verify-guards my-fact)

In [ ]:
(my-fact 10)

In [ ]:
; Prove: factorial always returns a positive integer
(defthm my-fact-positive
  (implies (natp n)
           (posp (my-fact n))))

## 8. The ACL2 Ecosystem

The ACL2 **community books** library contains over **145,000 verified theorems**
spanning virtually every area of computer science:

| Topic | ACL2 Books | Examples |
|-------|-----------|----------|
| **Sorting** | `books/sorting/` | Bubble, insertion, merge, quicksort with correctness proofs |
| **Graph Algorithms** | `books/misc/dijkstra-shortest-path` | Verified Dijkstra's with 126 theorems |
| **Data Structures** | `books/kestrel/data/treeset/` | BSTs, leftist trees, finite sets |
| **Machine Models** | `books/demos/marktoberdorf-08/m1.lisp` | JVM-like stack machines (M1–M6) |
| **Hardware** | `books/projects/fm9001/` | Complete verified processor (FM9001) |
| **Cryptography** | `books/kestrel/crypto/` | ChaCha20, ECDSA, R1CS |
| **SAT Solving** | `books/clause-processors/SULFA/` | Verified SAT solver integration |
| **Regex/Automata** | `books/projects/regex/` | POSIX regex engine with proofs |
| **Compiler** | `books/workshops/1999/compiler/` | Self-hosting verified compiler |
| **Concurrency** | `books/projects/aleo/bft/` | Byzantine fault tolerance proofs |

This course will explore many of these in the following modules.

## 9. Exercises

1. **Define `my-last`**: Write a function that returns the last element of a non-empty list.
   Prove that `(my-last (my-app x (list e)))` equals `e`.

2. **Define `my-count`**: Write a function that counts occurrences of element `e` in list `x`.
   Prove that `(my-count e (my-app x y))` equals `(+ (my-count e x) (my-count e y))`.

3. **Failing proof**: Try to prove `(equal (my-app x y) (my-app y x))`.
   Why does it fail? Can you find a counterexample manually?

4. **Guards**: Add a guard to `my-len` requiring `(true-listp x)` and verify it.

In [ ]:
; Exercise 1: Define my-last here


In [ ]:
; Exercise 2: Define my-count here


In [ ]:
; Exercise 3: Try the failing proof


In [ ]:
; Exercise 4: Add guards to my-len


## Summary

In this module, you learned:

- **S-expressions**: ACL2's prefix notation and data types (atoms, lists, cons pairs)
- **`defun`**: How to define total, terminating functions
- **`defthm`**: How to state and prove universal properties
- **Guards**: How to connect logical definitions to efficient execution
- **The verification workflow**: model → specify → prove → execute

**Key insight**: Every theorem ACL2 proves is **machine-checked**. Unlike testing
(which samples behavior) or code review (which depends on human attention), a formal
proof covers *all* cases. Once ACL2 says Q.E.D., the property holds **forever**.

**Next module →** Module 2: Lists, Recursion, and Induction Principles